In [27]:
from rdflib import Graph,URIRef,Namespace,BNode,Literal
from rdflib.namespace import XSD,RDF
import pandas as pd
import argparse
from collections import defaultdict
from datetime import datetime
import json
from datetime import datetime


In [28]:
def LoadGraph(directory):
    graph = Graph()
    print("Started loading graph...")
    graph.parse(directory, format="turtle",publicID="https://example.org/")
    print("Graph loaded successfully.")
    return graph

In [29]:
def SaveGraph(directory,final_graph):
    print('Started writing file to disk')
    final_graph.serialize(destination=directory, format="turtle")
    print('File written successfully')

In [30]:
def CreateSensorSet(graph): #if i need to carry any metadata for the sensor, I should query it here 
    sensor_set = set()
    get_sensor_query = ''' 
PREFIX sosa: <http://www.w3.org/ns/sosa/>

SELECT DISTINCT ?sensor
WHERE {
  ?s sosa:madeBySensor ?sensor .
}
'''
    print('Started identifying unique sensors')
    # store actual RDF terms (URIRef or Literal), not stringified values
    for sensor in graph.query(get_sensor_query):
        sensor_term = sensor[0]   # this is an rdflib term (URIRef or Literal)
        sensor_set.add(sensor_term)

    count = len(sensor_set)
    print(f'{count} Sensors identified successfully')
    return sensor_set


In [ ]:
def CreateTSS(sensor_set, graph):
    # Core RML and RDF Namespaces
    EX = Namespace('http://example.com/attributes/')
    OBS = Namespace('http://example.com/observations/')
    RDF = Namespace('http://www.w3.org/1999/02/22-rdf-syntax-ns#')
    RDFS = Namespace('http://www.w3.org/2000/01/rdf-schema#')
    RML = Namespace('http://w3id.org/rml/')
    SOSA = Namespace('http://www.w3.org/ns/sosa/')
    SSN = Namespace('http://www.w3.org/ns/ssn/')
    WATERINFO = Namespace('http://example.com/waterinfo/')
    XSD = Namespace('http://www.w3.org/2001/XMLSchema#')
    TSS = Namespace('https://w3id.org/tss#')

    base_snippet = Namespace("https://example.org/tss/snippet/")
    sensor_reading_id = Namespace("https://example.org/tss/snippet/reading/")
    final_graph = Graph()

    final_graph.bind('EX',EX)
    final_graph.bind('obs', OBS)
    final_graph.bind('rdf', RDF)
    final_graph.bind('rdfs', RDFS)
    final_graph.bind('rml', RML)
    final_graph.bind('sosa', SOSA)
    final_graph.bind('ssn', SSN)
    final_graph.bind('waterinfo', WATERINFO)
    final_graph.bind('xsd', XSD)
    final_graph.bind('tss',TSS)

    #print("Creating TSS graph...")
    print("started extracting observations per sensor")
    
    
    for i,sensor in enumerate (sensor_set):
        sensor_json_array = []
        print(f"started extracting observations for sensor {i+1}")

        # One SPARQL query per sensor (HUGE SPEEDUP)
        data_per_sensor_query = """
        PREFIX sosa: <http://www.w3.org/ns/sosa/>
        PREFIX ex: <http://example.com/attributes/>
        PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
        PREFIX waterinfo: <http://example.com/waterinfo/>

        SELECT ?OBSERVATION ?TIME ?READING ?qualityCode ?qualityName ?qualityDesc ?qualityColor ?interpolation
        WHERE {
            ?OBSERVATION a sosa:Observation ;
                        sosa:resultTime ?TIME ;
                        sosa:hasSimpleResult ?READING ;
                        sosa:madeBySensor ?currentSensor ;
                        ex:qualityCode ?qualityCode ;
                        ex:qualityCodeName ?qualityName ;
                        ex:qualityCodeDescription ?qualityDesc ;
                        ex:qualityCodeColor ?qualityColor ;
                        ex:interpolationType ?interpolation .
        }
        ORDER BY ?TIME
        """
        results = graph.query(data_per_sensor_query, initBindings={'currentSensor': sensor})
        
        observation_dimention_json = {
                "dimentions":[len(results),8] # 8 attributes per sensor reading
            }
        
        results_length = len(results)


        for J,row in enumerate(results):
            #UNIX time format conversion
            dt = datetime.fromisoformat(str(row.TIME))
            timestamp = dt.timestamp()

            #print(f"sensor: {sensor}    time:{row.TIME}     value:{row.READING}")
            observation_json_object = { #create json object per reading
                "id": URIRef(sensor_reading_id[f"{i}"]),
                "time": timestamp,
                "value": float(row.READING),
                "interpolationType": row.interpolation,
                "qualityCode":row.qualityCode,
                "qualityCodeColor":row.qualityColor,
                "qualityCodeDescription":row.qualityDesc,
                "qualityCodeName":row.qualityName

            }
           

            #-- Commenting out attribute names because they kinda feel redundant
            #sensor_json_array.append("id")
            sensor_json_array.append(URIRef(sensor_reading_id[f"{J}"]))
            #sensor_json_array.append("time")
            sensor_json_array.append(timestamp)
            #sensor_json_array.append("value")
            sensor_json_array.append(float(row.READING))
            #sensor_json_array.append("interpolation")


            # sensor_json_array.append(str(row.interpolation))
            # #sensor_json_array.append("qCode")
            # sensor_json_array.append(int(row.qualityCode))
            # #sensor_json_array.append("qColor")
            # sensor_json_array.append(str(row.qualityColor))
            # #sensor_json_array.append("qDesc")
            # sensor_json_array.append(str(row.qualityDesc))
            # #sensor_json_array.append("qName")
            # sensor_json_array.append(str(row.qualityName))

        
            #sensor_json_array.append(observation_json_object)  #append all readings per sensor
        json_output = json.dumps(sensor_json_array,indent=1)
        print(f"extracting observations for sensor {i+1} successfully finished")
        print(f"started creating RDF object for sensor number {i+1}")

        #for getting "from" and "to", we extract them from last sensor
        # 1. Convert to a list to enable indexing
        results_list = list(results)
        first_member = results_list[0]
        last_member = results_list[-1]
        #print(f"From: {first_member.TIME}   To: {last_member.TIME}")

        #Create a TSS ID per sensor
        snippet = URIRef(base_snippet[f"{sensor}"])
        template = BNode()

        #Create a TSS context per sensor'
        context_data = {
            "@context": {
                "id": "@id",
                "time": {
                    "@id": str(SOSA.resultTime),
                    "@type": str(XSD.integer)
                },
                "value": {
                    "@id": str(SOSA.hasSimpleResult),
                    "@type": str(XSD.double)
                },
                # "interpolationType": {
                #     "@id": str(EX.interpolationType),
                #     "@type": str(XSD.string)
                # },
                # "qualityCode": {
                #     "@id": str(EX.qualityCode),
                #     "@type": str(XSD.integer)
                # },
                # "qualityCodeColor": {
                #     "@id": str(EX.qualityCodeColor),
                #     "@type": str(XSD.string)
                # },
                # "qualityCodeDescription": {
                #     "@id": str(EX.qualityCodeDescription),
                #     "@type": str(XSD.string)
                # },
                # "qualityCodeName": {
                #     "@id": str(EX.qualityCodeName),
                #     "@type": str(XSD.string)
                # }
            }
        }


        context_data_dump = json.dumps(context_data,indent=1)

        final_graph.add((snippet, RDF.type, TSS.Snippet))
        final_graph.add((snippet,TSS["from"],first_member.TIME))
        final_graph.add((snippet,TSS["until"],last_member.TIME))
        final_graph.add((snippet, TSS.points, Literal(json_output,datatype=RDF.JSON)))
        final_graph.add((snippet, TSS.about, template))
        final_graph.add((template, RDF.type, TSS.PointTemplate))
        final_graph.add((template, SOSA.madeBySensor, sensor))
        final_graph.add((template, SOSA.observedProperty,WATERINFO.Conductivity ))
        final_graph.add((snippet,TSS.context,Literal(context_data_dump,datatype=RDF.JSON)))
        #final_graph.add((snippet,TSS.dimentions,Literal(observation_dimention_json,datatype=RDF.JSON)))
        #final_graph.add((snippet, TSS.rows, Literal(results_length, datatype=XSD.integer)))
        #final_graph.add((snippet, TSS.columns, Literal(8, datatype=XSD.integer)))
        
    
    print("TSS graph created.")
    return final_graph

In [32]:
print("Program started!")
Original_graph  = LoadGraph("../data/timeseriessample.ttl")
Sensor_set = CreateSensorSet(Original_graph)
final_graph = CreateTSS(Sensor_set,Original_graph)
SaveGraph("../data/TSSgraph (proposed modifications).ttl",final_graph)


Program started!
Started loading graph...
Graph loaded successfully.
Started identifying unique sensors
1 Sensors identified successfully
started extracting observations per sensor
started extracting observations for sensor 1
extracting observations for sensor 1 successfully finished
started creating RDF object for sensor number 1
TSS graph created.
Started writing file to disk
File written successfully
